# Skin Lesion CAD System: EfficientNetB0 Training on HAM10000

This notebook trains a deep convolutional neural network (**EfficientNetB0**) via transfer learning to classify skin lesions as **Benign** or **Malignant** using dermoscopy images from the HAM10000 dataset.

### Key Methodologies:
1. **Binary Mapping**: 7 HAM10000 classes mapped to Malignant (*mel*, *bcc*, *akiec*) vs Benign (*nv*, *bkl*, *df*, *vasc*).
2. **Grouped Splitting**: Splitting by `lesion_id` to eliminate patient data leakage between train/validation/test sets.
3. **Class Imbalance Handling**: Weighted loss calculation via `compute_class_weight` to address the 67% benign distribution.
4. **Two-Stage Fine-Tuning**: Head warmup followed by unfreezing upper convolution layers.
5. **Clinical Evaluation**: Focus on AUC-ROC and Malignant Recall/Sensitivity to minimize false negatives.

In [ ]:
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.model_selection import GroupShuffleSplit
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay, roc_auc_score, roc_curve

print("TensorFlow Version:", tf.__version__)
print("GPUs Available:", len(tf.config.list_physical_devices("GPU")))

## 1. Load Dataset Metadata & Map Labels

In [ ]:
IMG_SIZE = 224
BATCH_SIZE = 32
SEED = 42

# Configure paths (Kaggle default)
BASE_DIR = "/kaggle/input/skin-cancer-mnist-ham10000"
METADATA_PATH = os.path.join(BASE_DIR, "HAM10000_metadata.csv")
IMG_DIRS = [
    os.path.join(BASE_DIR, "HAM10000_images_part_1"),
    os.path.join(BASE_DIR, "HAM10000_images_part_2"),
    os.path.join(BASE_DIR, "ham10000_images_part_1"),
    os.path.join(BASE_DIR, "ham10000_images_part_2"),
]

df = pd.read_csv(METADATA_PATH)

MALIGNANT = {"mel", "bcc", "akiec"}
BENIGN = {"nv", "bkl", "df", "vasc"}

def map_label(dx):
    if dx in MALIGNANT:
        return "malignant"
    elif dx in BENIGN:
        return "benign"
    return None

df["label"] = df["dx"].apply(map_label)
df = df.dropna(subset=["label"]).copy()

def find_image_path(image_id):
    for d in IMG_DIRS:
        for ext in [".jpg", ".jpeg", ".png", ".JPG"]:
            p = os.path.join(d, image_id + ext)
            if os.path.exists(p):
                return p
    return None

df["image_path"] = df["image_id"].apply(find_image_path)
df = df.dropna(subset=["image_path"]).copy()
print(f"Total usable dermoscopic images: {len(df)}")
print(df["label"].value_counts())

## 2. Grouped Train / Validation / Test Splitting

Splitting by `lesion_id` ensures that multiple dermoscopic images of the same physical lesion do not cross over into test or validation partitions, preventing data leakage.

In [ ]:
gss_outer = GroupShuffleSplit(n_splits=1, test_size=0.30, random_state=SEED)
train_idx, temp_idx = next(gss_outer.split(df, groups=df["lesion_id"]))
train_df = df.iloc[train_idx].copy()
temp_df = df.iloc[temp_idx].copy()

gss_inner = GroupShuffleSplit(n_splits=1, test_size=0.50, random_state=SEED)
val_idx, test_idx = next(gss_inner.split(temp_df, groups=temp_df["lesion_id"]))
val_df = temp_df.iloc[val_idx].copy()
test_df = temp_df.iloc[test_idx].copy()

print(f"Train set: {len(train_df)} images")
print(f"Val set:   {len(val_df)} images")
print(f"Test set:  {len(test_df)} images")

## 3. Data Augmentation & Generators

In [ ]:
train_datagen = ImageDataGenerator(
    rescale=1.0 / 255.0,
    rotation_range=40,
    width_shift_range=0.15,
    height_shift_range=0.15,
    shear_range=0.15,
    zoom_range=0.20,
    horizontal_flip=True,
    vertical_flip=True,
    fill_mode="nearest",
)
val_test_datagen = ImageDataGenerator(rescale=1.0 / 255.0)

train_gen = train_datagen.flow_from_dataframe(
    train_df, x_col="image_path", y_col="label",
    target_size=(IMG_SIZE, IMG_SIZE), class_mode="binary",
    batch_size=BATCH_SIZE, seed=SEED
)
val_gen = val_test_datagen.flow_from_dataframe(
    val_df, x_col="image_path", y_col="label",
    target_size=(IMG_SIZE, IMG_SIZE), class_mode="binary",
    batch_size=BATCH_SIZE, seed=SEED, shuffle=False
)
test_gen = val_test_datagen.flow_from_dataframe(
    test_df, x_col="image_path", y_col="label",
    target_size=(IMG_SIZE, IMG_SIZE), class_mode="binary",
    batch_size=BATCH_SIZE, seed=SEED, shuffle=False
)

# Compute balanced class weights for training
labels = train_df["label"].map(train_gen.class_indices).values
class_weights_arr = compute_class_weight("balanced", classes=np.unique(labels), y=labels)
class_weights = dict(enumerate(class_weights_arr))
print("Class Indices:", train_gen.class_indices)
print("Class Weights:", class_weights)

with open("class_indices.json", "w") as f:
    json.dump(train_gen.class_indices, f, indent=2)

## 4. Build EfficientNetB0 Transfer Learning Model

In [ ]:
base_model = EfficientNetB0(include_top=False, weights="imagenet", input_shape=(IMG_SIZE, IMG_SIZE, 3))
base_model.trainable = False

inputs = layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3), name="input_layer")
x = base_model(inputs, training=False)
x = layers.GlobalAveragePooling2D(name="avg_pool")(x)
x = layers.Dropout(0.3, name="top_dropout_1")(x)
x = layers.Dense(128, activation="relu", name="dense_128")(x)
x = layers.Dropout(0.2, name="top_dropout_2")(x)
outputs = layers.Dense(1, activation="sigmoid", name="prediction_prob")(x)

model = models.Model(inputs, outputs, name="skin_cancer_efficientnetb0")
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="binary_crossentropy",
    metrics=["accuracy", tf.keras.metrics.AUC(name="auc"), tf.keras.metrics.Recall(name="recall"), tf.keras.metrics.Precision(name="precision")]
)
model.summary()

## 5. Two-Stage Training: Head Warmup & Fine-Tuning

In [ ]:
early_stop = callbacks.EarlyStopping(monitor="val_auc", mode="max", patience=4, restore_best_weights=True)
checkpoint = callbacks.ModelCheckpoint("best_checkpoint.keras", monitor="val_auc", mode="max", save_best_only=True)

print("--- Stage 1: Training Classification Head ---")
history_head = model.fit(
    train_gen, validation_data=val_gen, epochs=10,
    class_weight=class_weights, callbacks=[early_stop, checkpoint]
)

print("--- Stage 2: Fine-Tuning Top 30 EfficientNet Layers ---")
base_model.trainable = True
for layer in base_model.layers[:-30]:
    layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss="binary_crossentropy",
    metrics=["accuracy", tf.keras.metrics.AUC(name="auc"), tf.keras.metrics.Recall(name="recall"), tf.keras.metrics.Precision(name="precision")]
)
history_fine = model.fit(
    train_gen, validation_data=val_gen, epochs=15,
    class_weight=class_weights, callbacks=[early_stop, checkpoint]
)

model.save("skin_cancer_model.keras")
print("Saved final model as skin_cancer_model.keras")

## 6. Evaluation on Unseen Holdout Test Set & Plots

In [ ]:
test_gen.reset()
y_prob = model.predict(test_gen).ravel()
y_pred = (y_prob >= 0.5).astype(int)
y_true = test_gen.classes
target_names = list(train_gen.class_indices.keys())

print(classification_report(y_true, y_pred, target_names=target_names))
test_auc = roc_auc_score(y_true, y_prob)
print(f"Holdout Test ROC-AUC: {test_auc:.4f}")

# Plot ROC Curve
fpr, tpr, _ = roc_curve(y_true, y_prob)
plt.figure(figsize=(6, 5))
plt.plot(fpr, tpr, color="#1e3d59", lw=2, label="EfficientNetB0 (AUC = %0.3f)" % test_auc)
plt.plot([0, 1], [0, 1], color="gray", linestyle="--")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate (Malignant Recall)")
plt.title("ROC Curve")
plt.legend()
plt.grid(True, alpha=0.3)
plt.savefig("roc_curve.png", dpi=150)
plt.show()

# Confusion Matrix
cm = confusion_matrix(y_true, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=target_names)
disp.plot(cmap="Blues")
plt.title("Test Confusion Matrix")
plt.savefig("confusion_matrix.png", dpi=150)
plt.show()